In [ ]:
import kagglehub
import pandas as pd
import os
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

path = kagglehub.dataset_download("computingvictor/transactions-fraud-datasets")

files = [f for f in os.listdir(path) if f.endswith('.csv')]
# Change to load 'transactions_data.csv' which is typically larger
csv_path = os.path.join(path, 'transactions_data.csv')

# Loading the entire file
df = pd.read_csv(csv_path)

df.columns = [c.replace('.', '_').replace(' ', '_') for c in df.columns]

project_id = 'project-18e188c2-2177-4abb-b5d'
dataset_id = 'tpay_fraud_project'
table_id = f"{project_id}.{dataset_id}.transactions"

# Setting up BigQuery client (upload will be done in a separate step if desired)
client = bigquery.Client(project=project_id)
dataset = bigquery.Dataset(f"{project_id}.{dataset_id}")
client.create_dataset(dataset, exists_ok=True)

In [ ]:
try:
    # Wysyłka danych do BigQuery z użyciem BigQuery Storage Write API dla większej szybkości
    df.to_gbq(
        table_id, # Używamy pełnego table_id, np. 'project.dataset.table'
        project_id=project_id,
        if_exists='replace', # Użyj 'append' jeśli chcesz dodawać, 'replace' aby nadpisać
        use_bqstorage_api=True
    )
    print(f"Sukces, Dane są w BigQuery pod nazwą: {table_id}")
except Exception as e:
    print(f"Wystąpił błąd podczas wysyłania danych do BigQuery: {e}")